# Command-Line Interface

LangGOAP ships a `langgoap` CLI for quick planning without writing a full
Python script.  It loads actions, goals, and world state from Python
module references (`module:variable`) and supports multiple output formats.

| Command | Purpose |
|---------|---------|
| `langgoap plan` | Plan a sequence of actions and render the result |
| `langgoap actions` | List actions in a library |
| `langgoap explain` | Diagnose why a goal is unreachable |
| `langgoap visualize` | Render a pre-computed plan |
| `langgoap --version` | Show the installed version |

## Setup — Fixture Module

Write a small Python module to disk with actions, goals, and world state
that the CLI can import.  The CLI uses `module:variable` references, so
any importable module works.

In [1]:
fixture_code = '''
"""CLI demo fixture — a 3-action data pipeline."""
from langgoap import ActionSpec, GoalSpec

ACTIONS = [
    ActionSpec(
        name="gather",
        preconditions={},
        effects={"data_gathered": True},
        cost=1.0,
    ),
    ActionSpec(
        name="process",
        preconditions={"data_gathered": True},
        effects={"data_processed": True},
        cost=2.0,
    ),
    ActionSpec(
        name="report",
        preconditions={"data_processed": True},
        effects={"report_ready": True},
        cost=1.0,
    ),
]

GOAL = GoalSpec(conditions={"report_ready": True})

IMPOSSIBLE_GOAL = GoalSpec(conditions={"impossible_condition": True})

WORLD_STATE = {"existing": True}
'''

with open("_cli_demo.py", "w") as f:
    f.write(fixture_code)

print("Wrote _cli_demo.py")

Wrote _cli_demo.py


## `langgoap --version`

Verify the CLI is installed.

In [2]:
!langgoap --version

langgoap, version 0.1.0


## `langgoap actions` — List Actions

Show actions in table format (default) and JSON format.

In [3]:
!langgoap actions _cli_demo:ACTIONS

Name       Cost  Preconditions  Effects
---------------------------------------
gather      1.0  {}             {'data_gathered': True}
process     2.0  {'data_gathered': True}  {'data_processed': True}
report      1.0  {'data_processed': True}  {'report_ready': True}

3 action(s)


In [4]:
!langgoap actions _cli_demo:ACTIONS -f json

[
  {
    "name": "gather",
    "cost": 1.0,
    "preconditions": {},
    "effects": {
      "data_gathered": true
    }
  },
  {
    "name": "process",
    "cost": 2.0,
    "preconditions": {
      "data_gathered": true
    },
    "effects": {
      "data_processed": true
    }
  },
  {
    "name": "report",
    "cost": 1.0,
    "preconditions": {
      "data_processed": true
    },
    "effects": {
      "report_ready": true
    }
  }
]


## `langgoap plan` — Plan Actions

The `plan` command runs the GOAP planner and renders the result.
It accepts `module:ACTIONS` and `module:GOAL` references.

### ASCII format (default)

In [5]:
!langgoap plan _cli_demo:ACTIONS _cli_demo:GOAL

Plan (3 steps, cost=4)
├── [0] gather
├── [1] process  (after: [0])
├── [2] report  (after: [1])



### JSON format

In [6]:
!langgoap plan _cli_demo:ACTIONS _cli_demo:GOAL -f json

{
  "status": "goal_achieved",
  "actions": [
    "gather",
    "process",
    "report"
  ],
  "total_cost": 4.0,
  "steps": 3
}


### Mermaid format

In [7]:
!langgoap plan _cli_demo:ACTIONS _cli_demo:GOAL -f mermaid

%%{ init: { "flowchart": { "nodeSpacing": 5 } } }%%
flowchart TD
    classDef action fill:#f8f9fa,stroke:#4a90d9,stroke-width:2px,color:#1a1a2e
    classDef cost fill:#d4edda,stroke:#2d8a4e,stroke-width:1px,color:#155724,font-size:10px
    a0_gather["gather"]:::action
    subgraph sg_a1_process[" "]
        direction LR
        a1_process["process"]:::action
        c1(["fa:fa-dollar-sign 2"]):::cost
    end
    a2_report["report"]:::action
    style sg_a1_process fill:none,stroke:none
    a0_gather --> a1_process
    a1_process --> a2_report



### Output to file

Use `-o` to write the plan to a file instead of stdout.

In [8]:
!langgoap plan _cli_demo:ACTIONS _cli_demo:GOAL -f json -o plan.json

import json
with open("plan.json") as f:
    print(json.dumps(json.load(f), indent=2))

Plan written to plan.json


{
  "status": "goal_achieved",
  "actions": [
    "gather",
    "process",
    "report"
  ],
  "total_cost": 4.0,
  "steps": 3
}


## `langgoap explain` — Diagnose Unreachable Goals

When a goal is unreachable, `explain` performs reachability analysis
and reports which conditions no action can produce.

In [9]:
!langgoap explain _cli_demo:ACTIONS _cli_demo:IMPOSSIBLE_GOAL

Goal conditions: {'impossible_condition': True}
Start state: {}



Unreachable conditions (no action produces these):
  - impossible_condition

Suggestion: Goal condition(s) ['impossible_condition'] cannot be produced by any available action. Add an action whose effects include these keys, or check for typos in the goal conditions.


For comparison, running `explain` on an achievable goal confirms
all conditions are producible:

In [10]:
!langgoap explain _cli_demo:ACTIONS _cli_demo:GOAL

Goal conditions: {'report_ready': True}
Start state: {}

All goal conditions are producible by the action set.

Suggestion: The goal conditions appear theoretically reachable but A* found no valid action ordering. This may indicate circular precondition dependencies or conflicting action effects. Review action preconditions and effects for internal consistency.


## World State

Pass `-w module:variable` or `-w state.json` to supply an initial world
state.  The planner uses it as the starting point for A* search.

In [11]:
!langgoap plan _cli_demo:ACTIONS _cli_demo:GOAL -w _cli_demo:WORLD_STATE

Plan (3 steps, cost=4)
├── [0] gather
├── [1] process  (after: [0])
├── [2] report  (after: [1])



## Cleanup

In [12]:
import os

for f in ["_cli_demo.py", "plan.json"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed {f}")

Removed _cli_demo.py
Removed plan.json


## Summary

- **`langgoap plan`** — plan and render in ASCII, JSON, Mermaid, DOT, or Gantt.
- **`langgoap actions`** — list actions in table or JSON format.
- **`langgoap explain`** — diagnose unreachable goals with reachability analysis.
- **`langgoap visualize`** — render a pre-computed `Plan` object.
- Module references (`module:variable`) let you point the CLI at any
  importable Python module — no special config files needed.

Every command here is verified by
[`tests/test_cli.py`](../../tests/test_cli.py).